# weights

> Lay an exported PP-OCRv6 checkpoint onto MAX module weights.


In [ ]:
#| default_exp weights


#| export
Parameter order follows the module tree, which is the order the Paddle exporter emitted Conv, ConvTranspose, BatchNormalization, MatMul, and the decomposed LayerNorm scale and bias. Paddle linear weights are stored as `(in, out)`. MAX linear weights are `(out, in)`.


In [ ]:
#| hide
from nbdev import show_doc


In [ ]:
#| export
from __future__ import annotations

from pathlib import Path

import numpy as np

from solid_lens.onnx import Graph, Node, load


In [ ]:
#| export
def _skip_identity(nodes: list[Node], index: int) -> int:
    while index < len(nodes) and nodes[index].op_type == "Identity":
        index += 1
    return index


In [ ]:
#| export
def _vector(graph: Graph, node: Node) -> np.ndarray | None:
    "A 1-D learned vector on this node, such as a LayerNorm scale or bias."
    for name in node.inputs:
        if name not in graph.initializers:
            continue
        array = np.asarray(graph.initializers[name], dtype=np.float32)
        if array.ndim == 1 and array.size > 1:
            return np.ascontiguousarray(array.reshape(-1))
    return None


In [ ]:
#| export
def _bias_array(graph: Graph, nodes: list[Node], index: int) -> np.ndarray | None:
    nxt = _skip_identity(nodes, index + 1)
    if nxt >= len(nodes) or nodes[nxt].op_type != "Add":
        return None
    rhs = nodes[nxt].inputs[1]
    if rhs not in graph.initializers:
        return None
    array = np.asarray(graph.initializers[rhs], dtype=np.float32)
    if array.size <= 1 and array.ndim == 1:
        return None
    return np.ascontiguousarray(array.reshape(-1))


A bias is the next Add, skipping Identity nodes, whose right-hand side is a learned vector. A scalar add is not a bias. LayerNorm scale is the same kind of vector sitting on a Mul, and the bias is the Add that follows it.


In [ ]:
#| export
def arrays_from_graph(graph: Graph) -> list[np.ndarray]:
    "Learned tensors in module state-dict order."
    nodes = graph.nodes
    arrays: list[np.ndarray] = []
    for index, node in enumerate(nodes):
        if node.op_type in ("Conv", "ConvTranspose"):
            arrays.append(np.ascontiguousarray(graph.initializers[node.inputs[1]], dtype=np.float32))
            if len(node.inputs) > 2 and node.inputs[2]:
                bias = graph.initializers[node.inputs[2]]
                arrays.append(np.ascontiguousarray(np.asarray(bias, dtype=np.float32).reshape(-1)))
            else:
                bias = _bias_array(graph, nodes, index)
                if bias is not None:
                    arrays.append(bias)
        elif node.op_type == "BatchNormalization":
            for name in node.inputs[1:5]:
                arrays.append(
                    np.ascontiguousarray(np.asarray(graph.initializers[name], dtype=np.float32).reshape(-1))
                )
        elif node.op_type == "MatMul" and node.inputs[1] in graph.initializers:
            weight = np.asarray(graph.initializers[node.inputs[1]], dtype=np.float32)
            arrays.append(np.ascontiguousarray(weight.T))
            bias = _bias_array(graph, nodes, index)
            if bias is None:
                raise ValueError(f"{node.name} is missing a bias")
            arrays.append(bias)
        elif node.op_type == "Mul":
            # Exported LayerNorm is mean/var, then scale (Mul) and bias (Add).
            gamma = _vector(graph, node)
            if gamma is None:
                continue
            nxt = _skip_identity(nodes, index + 1)
            if nxt >= len(nodes) or nodes[nxt].op_type != "Add":
                raise ValueError(f"{node.name} scale is missing a LayerNorm bias")
            beta = _vector(graph, nodes[nxt])
            if beta is None:
                raise ValueError(f"{node.name} scale is missing a LayerNorm bias")
            arrays.append(gamma)
            arrays.append(beta)
    return arrays


Conv weights stay in the checkpoint's layout. `conv2d` was built with `permute=True` so that layout is already OIHW. MatMul is the exception: the stored matrix is `(in, out)` and MAX wants `(out, in)`, so it is transposed here.


In [ ]:
import numpy as np
from fastcore.test import test_eq
from solid_lens.onnx import Graph, Node

weight = np.arange(6, dtype=np.float32).reshape(2, 3)
graph = Graph()
graph.initializers["w"] = weight
graph.initializers["b"] = np.array([10, 20, 30], np.float32)
matmul = Node()
matmul.op_type = "MatMul"
matmul.name = "fc"
matmul.inputs = ["x", "w"]
add = Node()
add.op_type = "Add"
add.inputs = ["y", "b"]
graph.nodes = [matmul, add]
arrays = arrays_from_graph(graph)
test_eq(arrays[0].tolist(), weight.T.tolist())
test_eq(arrays[1].tolist(), [10.0, 20.0, 30.0])
arrays[0]


array([[0., 3.],
       [1., 4.],
       [2., 5.]], dtype=float32)

In [ ]:
#| export
def checkpoint_arrays(path: Path) -> list[np.ndarray]:
    "Learned tensors in an ONNX checkpoint, in module state-dict order."
    return arrays_from_graph(load(path))


In [ ]:
#| export
def _weights_in_definition_order(module, prefix: str = ""):
    "Depth-first weights. This matches the order the exporter wrote them."
    for weight_name, weight in module.layer_weights.items():
        if weight_name in module._shared_weights:
            continue
        yield f"{prefix}{weight_name}", weight
    for local_name, child in module.sublayers.items():
        child_prefix = prefix if child._omit_module_attr_name else f"{prefix}{local_name}."
        yield from _weights_in_definition_order(child, child_prefix)


MAX walks `layer_weights` and then `sublayers`. Defining a submodule later in `__init__` puts its tensors later. That is the same order as the exporter, which is why the layer notebooks keep the attribute order they do.


In [ ]:
#| export
def load_checkpoint(module, path: Path) -> None:
    arrays = checkpoint_arrays(path)
    slots = list(_weights_in_definition_order(module))
    if len(arrays) != len(slots):
        raise ValueError(
            f"{path.name}: checkpoint has {len(arrays)} tensors, the MAX module has {len(slots)}"
        )
    state = {}
    for (name, weight), array in zip(slots, arrays, strict=True):
        expected = tuple(int(dim) for dim in weight.shape)
        if array.shape != expected:
            raise ValueError(f"{name}: checkpoint shape {array.shape} != module shape {expected}")
        state[name] = array
    module.load_state_dict(state, strict=True)


A count mismatch and a shape mismatch are both errors. There is no partial load. If a new layer is inserted in the wrong place, this is the function that says so.
